# Stage 5B — Corrected Motion-Compensated DINOv2 Token Matching

This notebook is a **corrected rerun** of the Stage 5B notebook you executed.

## What changed

The previous notebook converted Stage 5A's 640×640 flow coordinates with:

`(patch_center + flow) * (630/640)`

That also scaled the **absolute patch-center coordinate**, which caused artificial spatial shifts even when optical flow was near zero. For example, the previous run reported about 50–73% "moved patches" in several videos even at very small mean motion.

This notebook fixes that coordinate conversion:

- keep the DINO patch center in 630×630 coordinates
- scale **only the optical-flow displacement** from 640×640 to 630×630
- convert the displacement to the nearest DINO patch shift
- with zero flow, the mapping must be exactly identity

It also avoids recomputing DINO embeddings twice for every frame pair: each sampled frame is embedded once and reused for the next pair.

## Previous run retained as a baseline

The executed notebook produced 683 frame pairs with:

- same-position similarity: **0.8736**
- motion-compensated similarity: **0.8327**
- similarity change: **−0.0409**
- same-position reuse: **79.94%**
- motion-compensated reuse: **72.06%**
- reuse change: **−7.88 percentage points**

Those numbers are retained only as the **previous implementation baseline**. This rerun is intended to test the corrected coordinate mapping.

The experiment measures token correspondence/reuse potential. It does **not** by itself prove end-to-end speedup, FLOP reduction, latency reduction, or downstream accuracy preservation.


In [19]:
# Install/update required packages
!pip install -q -U huggingface_hub transformers

print("Packages ready.")


Packages ready.


In [20]:
# Imports and Hugging Face authentication
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")

if not hf_token:
    raise RuntimeError("HF_TOKEN is empty or missing from Kaggle Secrets.")

login(token=hf_token)
print("Authenticated as:", whoami(token=hf_token)["name"])


Authenticated as: theramirey


In [21]:
# Configuration
from pathlib import Path
import json
import time
import zipfile
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn.functional as F

REPO_ID = "nexar-ai/nexar_collision_prediction"
REPO_TYPE = "dataset"

WORK_DIR = Path("/kaggle/working/nexar_stage5")
VIDEO_DIR = WORK_DIR / "videos"
OUTPUT_DIR = WORK_DIR / "stage5B_corrected_results"

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FPS = 5.0

# Stage 5A used 640x640 for optical flow.
IMAGE_SIZE = 640

# DINOv2 ViT-S/14 is evaluated at exact 630x630 so that:
# 630 / 14 = 45 patches per dimension.
DINO_SIZE = 630
PATCH_SIZE = 14
GRID_SIZE = 45
NUM_PATCHES = GRID_SIZE * GRID_SIZE

NUM_VIDEOS = 4

# Same dynamic-threshold rule as the previous notebook.
BASE_THRESHOLD = 0.75
MOTION_ALPHA = 0.10
MIN_THRESHOLD = 0.70
MAX_THRESHOLD = 0.90

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

print("DINO input:", DINO_SIZE, "x", DINO_SIZE)
print("Patch grid:", GRID_SIZE, "x", GRID_SIZE)
print("Patch tokens:", NUM_PATCHES)


Device: cuda
GPU: Tesla T4
DINO input: 630 x 630
Patch grid: 45 x 45
Patch tokens: 2025


## 1. Load and verify Stage 5A motion vectors

The Stage 5A file contains median Farneback flow per 14×14 patch, stored as `[dx, dy]` on a 45×45 grid.


In [22]:
# Find Stage 5A motion vectors
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
MOTION_FILENAME = "stage5_patch_motion_vectors.npz"

matches = list(INPUT_ROOT.rglob(MOTION_FILENAME))
if not matches:
    matches = list(WORK_ROOT.rglob(MOTION_FILENAME))

if matches:
    motion_file = matches[0]
else:
    zip_matches = list(INPUT_ROOT.rglob("Stage5A_Motion_Results.zip"))
    if not zip_matches:
        zip_matches = list(WORK_ROOT.rglob("Stage5A_Motion_Results.zip"))
    if not zip_matches:
        zip_matches = [
            p for p in INPUT_ROOT.rglob("*.zip")
            if "stage5a" in p.name.lower()
        ]

    if not zip_matches:
        raise FileNotFoundError(
            "Could not find stage5_patch_motion_vectors.npz or Stage5A_Motion_Results.zip."
        )

    zip_path = zip_matches[0]
    extract_dir = WORK_ROOT / "stage5A_input"
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

    extracted = list(extract_dir.rglob(MOTION_FILENAME))
    if not extracted:
        raise FileNotFoundError(
            f"{MOTION_FILENAME} was not found inside {zip_path}."
        )
    motion_file = extracted[0]

print("Stage 5A motion file:")
print(motion_file)


Stage 5A motion file:
/kaggle/input/datasets/theramiway/stage5a-motion-results/Stage5A_Motion_Results/stage5_patch_motion_vectors.npz


In [23]:
# Verify Stage 5A arrays
motion_npz = np.load(motion_file)
motion_arrays = {}

print("=" * 70)
print("STAGE 5A MOTION VECTOR VERIFICATION")
print("=" * 70)

for key in motion_npz.files:
    arr = motion_npz[key]

    if arr.ndim != 4 or arr.shape[1:] != (GRID_SIZE, GRID_SIZE, 2):
        raise ValueError(
            f"{key}: expected [pairs,45,45,2], got {arr.shape}"
        )

    if not np.isfinite(arr).all():
        raise ValueError(f"{key} contains NaN/Inf.")

    motion_arrays[key] = arr.astype(np.float32)

    print(f"{key}: shape={arr.shape}, dtype={arr.dtype}")

required_ids = {"01040", "01041", "01042", "01043"}
if set(motion_arrays.keys()) != required_ids:
    raise ValueError(
        f"Expected motion vectors for {sorted(required_ids)}, "
        f"found {sorted(motion_arrays.keys())}"
    )

print("Total motion pairs:", sum(len(x) for x in motion_arrays.values()))
print("✓ Stage 5A arrays are valid.")


STAGE 5A MOTION VECTOR VERIFICATION
01040: shape=(196, 45, 45, 2), dtype=float32
01041: shape=(197, 45, 45, 2), dtype=float32
01042: shape=(89, 45, 45, 2), dtype=float32
01043: shape=(201, 45, 45, 2), dtype=float32
Total motion pairs: 683
✓ Stage 5A arrays are valid.


## 2. Find/download the same four Nexar videos

The notebook first checks Kaggle input/working. If a video is absent, it downloads the same gated training video through Hugging Face using `HF_TOKEN`.


In [24]:
# Discover accessible Nexar MP4s
from huggingface_hub import list_repo_files, hf_hub_download

repo_files = list_repo_files(
    REPO_ID,
    repo_type=REPO_TYPE,
    token=hf_token
)

mp4s = sorted(f for f in repo_files if f.lower().endswith(".mp4"))
negative_train = [
    f for f in mp4s
    if f.startswith("train/negative/")
]

print("Accessible MP4 files:", len(mp4s))
print("Train/negative videos:", len(negative_train))


Accessible MP4 files: 2844
Train/negative videos: 750


In [25]:
# Find the exact four Stage 5A videos locally first
VIDEO_IDS = set(motion_arrays.keys())

found_videos = {}

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    if not root.exists():
        continue

    for p in root.rglob("*.mp4"):
        if p.stem in VIDEO_IDS and p.stat().st_size > 0:
            found_videos.setdefault(p.stem, p)

print("Videos already available:")
for vid in sorted(VIDEO_IDS):
    if vid in found_videos:
        print(" ✓", vid, "->", found_videos[vid])
    else:
        print(" ✗", vid, "NOT FOUND")

missing_ids = sorted(VIDEO_IDS - set(found_videos))

if missing_ids:
    print("\nDownloading missing videos from Hugging Face...")

    selected_remote = [
        f for f in negative_train
        if Path(f).stem in missing_ids
    ]

    for remote_path in selected_remote:
        video_id = Path(remote_path).stem
        local_path = VIDEO_DIR / remote_path
        local_path.parent.mkdir(parents=True, exist_ok=True)

        if local_path.exists() and local_path.stat().st_size > 0:
            found_videos[video_id] = local_path
            print("Already available:", local_path)
            continue

        print("\nDownloading:", remote_path)

        downloaded_path = hf_hub_download(
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            filename=remote_path,
            token=hf_token,
            local_dir=str(VIDEO_DIR)
        )

        found_videos[video_id] = Path(downloaded_path)
        print("✓ Downloaded:", downloaded_path)

missing_after = sorted(VIDEO_IDS - set(found_videos))

if missing_after:
    raise RuntimeError(
        "Could not obtain: " + ", ".join(missing_after)
    )

video_lookup = found_videos
print("\n✓ All four videos are available.")


Videos already available:
 ✓ 01040 -> /kaggle/working/nexar_stage5/videos/train/negative/01040.mp4
 ✓ 01041 -> /kaggle/working/nexar_stage5/videos/train/negative/01041.mp4
 ✓ 01042 -> /kaggle/working/nexar_stage5/videos/train/negative/01042.mp4
 ✓ 01043 -> /kaggle/working/nexar_stage5/videos/train/negative/01043.mp4

✓ All four videos are available.


In [26]:
# Verify all four videos
video_metadata = {}

print("=" * 70)
print("VIDEO VERIFICATION")
print("=" * 70)

for video_id in sorted(VIDEO_IDS):
    path = Path(video_lookup[video_id])
    cap = cv2.VideoCapture(str(path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open {path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frames / fps if fps > 0 else 0.0

    cap.release()

    video_metadata[video_id] = {
        "path": str(path),
        "fps": fps,
        "frames": frames,
        "width": width,
        "height": height,
        "duration_sec": duration
    }

    print(f"\n{video_id}.mp4")
    print(" FPS:", fps)
    print(" Frames:", frames)
    print(" Resolution:", f"{width} x {height}")
    print(" Duration:", f"{duration:.2f} sec")


VIDEO VERIFICATION

01040.mp4
 FPS: 30.0
 Frames: 1180
 Resolution: 1280 x 720
 Duration: 39.33 sec

01041.mp4
 FPS: 30.0
 Frames: 1183
 Resolution: 1280 x 720
 Duration: 39.43 sec

01042.mp4
 FPS: 30.0
 Frames: 540
 Resolution: 1280 x 720
 Duration: 18.00 sec

01043.mp4
 FPS: 30.0
 Frames: 1211
 Resolution: 1280 x 720
 Duration: 40.37 sec


## 3. Load DINOv2 ViT-S/14

We keep the same pretrained model as the previous notebook.


In [27]:
from transformers import AutoImageProcessor, AutoModel
from PIL import Image

MODEL_NAME = "facebook/dinov2-small"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print("Loaded:", MODEL_NAME)
print("Hidden size:", model.config.hidden_size)
print("Patch size:", model.config.patch_size)


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loaded: facebook/dinov2-small
Hidden size: 384
Patch size: 14


In [28]:
@torch.no_grad()
def get_patch_embeddings(rgb_frame):
    # Exact 630x630 input -> 45x45 DINO patches.
    frame = cv2.resize(
        rgb_frame,
        (DINO_SIZE, DINO_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    image = Image.fromarray(frame)

    inputs = processor(
        images=image,
        return_tensors="pt",
        do_resize=False,
        do_center_crop=False
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    outputs = model(**inputs)

    # Remove CLS token.
    tokens = outputs.last_hidden_state[:, 1:, :]

    if tokens.shape[1] != NUM_PATCHES:
        raise RuntimeError(
            f"DINOv2 returned {tokens.shape[1]} patch tokens; "
            f"expected {NUM_PATCHES}."
        )

    # Cosine similarity becomes a dot product after normalization.
    tokens = F.normalize(tokens, p=2, dim=-1)

    return tokens[0].cpu().numpy().astype(np.float32)


def read_frame_at_index(cap, frame_index):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_index))
    ok, frame = cap.read()

    if not ok:
        return None

    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)


## 4. Correct motion-to-patch mapping

### The important correction

Stage 5A flow is measured in **640×640 pixel coordinates**. DINO is evaluated at **630×630**.

The old implementation scaled the absolute patch center:

`(x + dx) * (630/640)`

That is incorrect because it compresses the patch-center grid itself.

The corrected mapping is:

`target_center = source_center + dx * (630/640)`

and then the displacement is converted to the nearest DINO patch.

This means **zero flow produces exactly zero patch displacement**.


In [29]:
def build_motion_target_indices_corrected(patch_flow):
    """
    Convert Stage 5A patch flow from 640x640 pixel displacement
    into DINO 630x630 patch displacement.

    patch_flow[..., 0] = dx
    patch_flow[..., 1] = dy

    IMPORTANT:
    We scale only dx/dy, not the absolute DINO patch centers.
    """
    scale = DINO_SIZE / IMAGE_SIZE

    dx_dino = patch_flow[..., 0] * scale
    dy_dino = patch_flow[..., 1] * scale

    # Convert pixel displacement to nearest patch displacement.
    dc = np.rint(dx_dino / PATCH_SIZE).astype(np.int32)
    dr = np.rint(dy_dino / PATCH_SIZE).astype(np.int32)

    rows, cols = np.indices((GRID_SIZE, GRID_SIZE))

    target_r = np.clip(rows + dr, 0, GRID_SIZE - 1)
    target_c = np.clip(cols + dc, 0, GRID_SIZE - 1)

    targets = target_r * GRID_SIZE + target_c

    return targets.reshape(-1).astype(np.int32)


In [30]:
# Mandatory identity sanity test
zero_flow = np.zeros((GRID_SIZE, GRID_SIZE, 2), dtype=np.float32)
zero_targets = build_motion_target_indices_corrected(zero_flow)

identity = np.arange(NUM_PATCHES, dtype=np.int32)

print("Zero-flow mapping differences:", np.sum(zero_targets != identity))

if not np.array_equal(zero_targets, identity):
    raise RuntimeError(
        "FAIL: zero-flow mapping is not identity. "
        "Motion compensation mapping is still incorrect."
    )

print("✓ Zero-flow mapping is exactly identity.")


Zero-flow mapping differences: 0
✓ Zero-flow mapping is exactly identity.


In [31]:
# Compare how many patches the corrected mapping moves.
# This is a diagnostic only; it is not a token-reuse percentage.

mapping_diagnostics = []

for video_id, motion in motion_arrays.items():
    moved_values = []

    for pair_flow in motion:
        targets = build_motion_target_indices_corrected(pair_flow)
        moved_values.append(
            np.mean(targets != np.arange(NUM_PATCHES)) * 100.0
        )

    mapping_diagnostics.append({
        "video": video_id,
        "mean_corrected_moved_patch_percent": float(np.mean(moved_values)),
        "median_corrected_moved_patch_percent": float(np.median(moved_values))
    })

mapping_diagnostics_df = pd.DataFrame(mapping_diagnostics)

print("Corrected motion-mapping diagnostic:")
display(mapping_diagnostics_df)


Corrected motion-mapping diagnostic:


,video,mean_corrected_moved_patch_percent,median_corrected_moved_patch_percent
0,01040,7.377929,3.506173
1,01041,28.715172,32.444444
2,01042,3.404633,3.358025
3,01043,23.716479,26.024691


## 5. Dynamic threshold

The threshold rule is kept from the previous experiment so that the rerun changes the motion mapping rather than changing every experimental variable at once.

For each patch:

`threshold = BASE + ALPHA × normalized_motion`

where motion is normalized by the pair's 90th-percentile motion and clipped to the configured minimum/maximum.


In [32]:
def dynamic_threshold(motion_magnitude):
    scale = np.percentile(motion_magnitude, 90)

    if scale < 1e-6:
        normalized = np.zeros_like(
            motion_magnitude,
            dtype=np.float32
        )
    else:
        normalized = np.clip(
            motion_magnitude / scale,
            0.0,
            1.0
        )

    threshold = BASE_THRESHOLD + MOTION_ALPHA * normalized

    return np.clip(
        threshold,
        MIN_THRESHOLD,
        MAX_THRESHOLD
    ).astype(np.float32)


## 6. Corrected Stage 5B processing

This version processes each sampled frame only once:

- first sampled frame → DINO
- next sampled frame → DINO
- compare them
- reuse the current embedding as the next previous embedding

This cuts DINO model calls roughly in half compared with the previous notebook.


In [33]:
def process_video_matching_corrected(video_path, motion_key):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open {video_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    stride = int(round(fps / TARGET_FPS))
    sampled_indices = list(range(0, total_frames, stride))

    motion = motion_arrays[motion_key]

    max_pairs = min(
        len(sampled_indices) - 1,
        len(motion)
    )

    if max_pairs <= 0:
        cap.release()
        raise RuntimeError(f"No frame pairs available for {video_path}")

    print(
        f"\n{video_path.name}: "
        f"{max_pairs} frame pairs"
    )

    # Embed the first frame once.
    prev_frame = read_frame_at_index(
        cap,
        sampled_indices[0]
    )

    if prev_frame is None:
        cap.release()
        raise RuntimeError(
            f"Could not read first sampled frame of {video_path}"
        )

    prev_tokens = get_patch_embeddings(prev_frame)

    records = []

    for pair_idx in range(max_pairs):
        curr_idx = sampled_indices[pair_idx + 1]

        curr_frame = read_frame_at_index(
            cap,
            curr_idx
        )

        if curr_frame is None:
            print("Warning: could not read frame", curr_idx)
            continue

        curr_tokens = get_patch_embeddings(curr_frame)

        patch_flow = motion[pair_idx]

        motion_mag = np.sqrt(
            patch_flow[..., 0] ** 2 +
            patch_flow[..., 1] ** 2
        ).reshape(-1)

        # Baseline: same spatial position.
        same_similarity = np.sum(
            prev_tokens * curr_tokens,
            axis=1
        )

        # Corrected motion-compensated target.
        targets = build_motion_target_indices_corrected(
            patch_flow
        )

        compensated_similarity = np.sum(
            prev_tokens * curr_tokens[targets],
            axis=1
        )

        thresholds = dynamic_threshold(motion_mag)

        same_reuse = same_similarity >= thresholds
        compensated_reuse = compensated_similarity >= thresholds

        improvement = (
            compensated_similarity -
            same_similarity
        )

        moved_position = (
            targets != np.arange(NUM_PATCHES)
        )

        records.append({
            "video": video_path.name,
            "pair_idx": pair_idx,
            "time_sec": curr_idx / fps,

            "mean_motion_px": float(motion_mag.mean()),
            "median_motion_px": float(np.median(motion_mag)),

            "mean_same_similarity": float(
                same_similarity.mean()
            ),

            "mean_motion_comp_similarity": float(
                compensated_similarity.mean()
            ),

            "mean_similarity_improvement": float(
                improvement.mean()
            ),

            "same_position_reuse_percent": float(
                same_reuse.mean() * 100
            ),

            "motion_comp_reuse_percent": float(
                compensated_reuse.mean() * 100
            ),

            "reuse_gain_percent": float(
                compensated_reuse.mean() * 100 -
                same_reuse.mean() * 100
            ),

            "mean_dynamic_threshold": float(
                thresholds.mean()
            ),

            "corrected_moved_patch_percent": float(
                moved_position.mean() * 100
            )
        })

        # IMPORTANT: current embedding becomes previous embedding.
        prev_tokens = curr_tokens

    cap.release()

    return pd.DataFrame(records)


In [34]:
# Run corrected Stage 5B
from tqdm.auto import tqdm

all_results = []
start_time = time.time()

for motion_key in sorted(motion_arrays.keys()):
    df = process_video_matching_corrected(
        video_lookup[motion_key],
        motion_key
    )

    if len(df):
        all_results.append(df)

if not all_results:
    raise RuntimeError("No corrected Stage 5B results were generated.")

stage5b_df = pd.concat(
    all_results,
    ignore_index=True
)

runtime = time.time() - start_time

print("\nCorrected Stage 5B frame pairs:", len(stage5b_df))
print("Runtime:", round(runtime, 2), "sec")



01040.mp4: 196 frame pairs

01041.mp4: 197 frame pairs

01042.mp4: 89 frame pairs

01043.mp4: 201 frame pairs

Corrected Stage 5B frame pairs: 683
Runtime: 271.24 sec


## 7. Aggregate results


In [35]:
per_video = (
    stage5b_df
    .groupby("video")
    .agg(
        pairs=("pair_idx", "count"),
        mean_motion_px=("mean_motion_px", "mean"),
        mean_same_similarity=("mean_same_similarity", "mean"),
        mean_motion_comp_similarity=(
            "mean_motion_comp_similarity",
            "mean"
        ),
        mean_similarity_improvement=(
            "mean_similarity_improvement",
            "mean"
        ),
        same_position_reuse_percent=(
            "same_position_reuse_percent",
            "mean"
        ),
        motion_comp_reuse_percent=(
            "motion_comp_reuse_percent",
            "mean"
        ),
        reuse_gain_percent=(
            "reuse_gain_percent",
            "mean"
        ),
        mean_dynamic_threshold=(
            "mean_dynamic_threshold",
            "mean"
        ),
        corrected_moved_patch_percent=(
            "corrected_moved_patch_percent",
            "mean"
        )
    )
    .reset_index()
)

overall = {
    "stage": "Stage 5B - Corrected Motion-Compensated DINOv2 Matching",
    "dataset": "Nexar Collision Prediction",
    "videos": int(stage5b_df["video"].nunique()),
    "frame_pairs": int(len(stage5b_df)),
    "dino_model": MODEL_NAME,
    "target_fps": TARGET_FPS,
    "dino_input": [DINO_SIZE, DINO_SIZE],
    "stage5a_flow_size": [IMAGE_SIZE, IMAGE_SIZE],
    "patch_size": PATCH_SIZE,
    "patch_grid": [GRID_SIZE, GRID_SIZE],
    "num_patches": NUM_PATCHES,

    "mean_same_position_similarity": float(
        stage5b_df["mean_same_similarity"].mean()
    ),

    "mean_motion_comp_similarity": float(
        stage5b_df["mean_motion_comp_similarity"].mean()
    ),

    "mean_similarity_improvement": float(
        stage5b_df["mean_similarity_improvement"].mean()
    ),

    "mean_same_position_reuse_percent": float(
        stage5b_df["same_position_reuse_percent"].mean()
    ),

    "mean_motion_comp_reuse_percent": float(
        stage5b_df["motion_comp_reuse_percent"].mean()
    ),

    "mean_reuse_gain_percent": float(
        stage5b_df["reuse_gain_percent"].mean()
    ),

    "mean_corrected_moved_patch_percent": float(
        stage5b_df["corrected_moved_patch_percent"].mean()
    ),

    "mean_dynamic_threshold": float(
        stage5b_df["mean_dynamic_threshold"].mean()
    ),

    "threshold": {
        "base": BASE_THRESHOLD,
        "motion_alpha": MOTION_ALPHA,
        "min": MIN_THRESHOLD,
        "max": MAX_THRESHOLD
    },

    "runtime_seconds": float(runtime),

    "correction": {
        "previous_mapping": "(source_center + flow) * (630/640)",
        "corrected_mapping": "source_center + flow * (630/640)",
        "target_assignment": "nearest 14x14 DINO patch",
        "zero_flow_identity_verified": True
    }
}

print("=" * 75)
print("CORRECTED STAGE 5B SUMMARY")
print("=" * 75)

print("Videos:", overall["videos"])
print("Frame pairs:", overall["frame_pairs"])
print(
    "Same-position similarity:",
    round(overall["mean_same_position_similarity"], 4)
)
print(
    "Motion-compensated similarity:",
    round(overall["mean_motion_comp_similarity"], 4)
)
print(
    "Similarity improvement:",
    round(overall["mean_similarity_improvement"], 4)
)
print(
    "Same-position reuse:",
    round(overall["mean_same_position_reuse_percent"], 2),
    "%"
)
print(
    "Motion-compensated reuse:",
    round(overall["mean_motion_comp_reuse_percent"], 2),
    "%"
)
print(
    "Reuse gain:",
    round(overall["mean_reuse_gain_percent"], 2),
    "percentage points"
)
print(
    "Corrected moved patches:",
    round(overall["mean_corrected_moved_patch_percent"], 2),
    "%"
)
print(
    "Mean dynamic threshold:",
    round(overall["mean_dynamic_threshold"], 4)
)

print("\nPer-video results:")
display(per_video)


CORRECTED STAGE 5B SUMMARY
Videos: 4
Frame pairs: 683
Same-position similarity: 0.8736
Motion-compensated similarity: 0.8845
Similarity improvement: 0.0109
Same-position reuse: 79.94 %
Motion-compensated reuse: 82.64 %
Reuse gain: 2.7 percentage points
Corrected moved patches: 17.82 %
Mean dynamic threshold: 0.7781

Per-video results:


,video,pairs,mean_motion_px,mean_same_similarity,mean_motion_comp_similarity,mean_similarity_improvement,same_position_reuse_percent,motion_comp_reuse_percent,reuse_gain_percent,mean_dynamic_threshold,corrected_moved_patch_percent
0,01040.mp4,196,1.612067,0.910174,0.913471,0.003297,88.311413,89.353238,1.041824,0.772605,7.377929
1,01041.mp4,197,6.025444,0.852687,0.874474,0.021788,73.596791,78.991038,5.394247,0.789598,28.715172
2,01042.mp4,89,0.915193,0.888964,0.889872,0.000908,86.737134,87.007352,0.270218,0.766418,3.404633
3,01043.mp4,201,5.317310,0.851693,0.863828,0.012136,74.968368,77.722253,2.753885,0.777220,23.716479


## 8. High-motion analysis

This separates the pairs where the average patch motion is substantial. It is useful because motion compensation is expected to matter most when motion is large.

This is descriptive analysis, not a claim that a particular threshold is optimal.


In [36]:
# High-motion analysis
HIGH_MOTION_THRESHOLD_PX = 10.0

high_motion = stage5b_df[
    stage5b_df["mean_motion_px"] >= HIGH_MOTION_THRESHOLD_PX
].copy()

print(
    f"High-motion pairs (mean motion >= {HIGH_MOTION_THRESHOLD_PX}px):",
    len(high_motion),
    "/",
    len(stage5b_df)
)

if len(high_motion):
    print(
        "Same-position similarity:",
        round(high_motion["mean_same_similarity"].mean(), 4)
    )
    print(
        "Motion-compensated similarity:",
        round(
            high_motion["mean_motion_comp_similarity"].mean(),
            4
        )
    )
    print(
        "Reuse gain:",
        round(
            high_motion["reuse_gain_percent"].mean(),
            2
        ),
        "percentage points"
    )


High-motion pairs (mean motion >= 10.0px): 29 / 683
Same-position similarity: 0.7873
Motion-compensated similarity: 0.8386
Reuse gain: 12.03 percentage points


## 9. Save CSV/JSON results


In [37]:
pair_csv = OUTPUT_DIR / "stage5B_corrected_pair_results.csv"
video_csv = OUTPUT_DIR / "stage5B_corrected_per_video.csv"
summary_json = OUTPUT_DIR / "stage5B_corrected_summary.json"
config_json = OUTPUT_DIR / "stage5B_corrected_config.json"

stage5b_df.to_csv(pair_csv, index=False)
per_video.to_csv(video_csv, index=False)

with open(summary_json, "w") as f:
    json.dump(overall, f, indent=2)

with open(config_json, "w") as f:
    json.dump(
        {
            "model": MODEL_NAME,
            "target_fps": TARGET_FPS,
            "stage5a_image_size": IMAGE_SIZE,
            "dino_size": DINO_SIZE,
            "patch_size": PATCH_SIZE,
            "grid_size": GRID_SIZE,
            "threshold": {
                "base": BASE_THRESHOLD,
                "motion_alpha": MOTION_ALPHA,
                "min": MIN_THRESHOLD,
                "max": MAX_THRESHOLD
            },
            "motion_mapping": {
                "flow_scale": DINO_SIZE / IMAGE_SIZE,
                "method": "scale flow displacement only, then nearest patch shift",
                "zero_flow_identity_verified": True
            },
            "stage5a_motion_file": str(motion_file)
        },
        f,
        indent=2
    )

print("Saved:")
print(pair_csv)
print(video_csv)
print(summary_json)
print(config_json)


Saved:
/kaggle/working/nexar_stage5/stage5B_corrected_results/stage5B_corrected_pair_results.csv
/kaggle/working/nexar_stage5/stage5B_corrected_results/stage5B_corrected_per_video.csv
/kaggle/working/nexar_stage5/stage5B_corrected_results/stage5B_corrected_summary.json
/kaggle/working/nexar_stage5/stage5B_corrected_results/stage5B_corrected_config.json


## 10. Create one downloadable ZIP


In [38]:
zip_path = Path(
    "/kaggle/working/Stage5B_Corrected_Motion_Compensated_Results.zip"
)

files_to_zip = [
    pair_csv,
    video_csv,
    summary_json,
    config_json
]

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(
                file,
                arcname=f"Stage5B_Corrected_Results/{file.name}"
            )

print("ZIP created:")
print(zip_path)
print(
    f"Size: {zip_path.stat().st_size / (1024**2):.2f} MB"
)


ZIP created:
/kaggle/working/Stage5B_Corrected_Motion_Compensated_Results.zip
Size: 0.05 MB


# Interpretation notes

### What to report

The primary comparison is still:

**same-position token matching vs corrected motion-compensated token matching.**

Report:
- mean cosine similarity
- mean reuse percentage
- reuse gain/loss in percentage points
- corrected moved-patch percentage
- per-video results
- high-motion subset results

### Important

Do not describe token reuse as actual compute reduction unless an actual selective-inference implementation is benchmarked.

Do not describe the dynamic threshold as optimal. It is an experimental rule retained from the previous run.

The previous Stage 5B result should be treated as the **pre-correction baseline**. The new notebook includes an explicit zero-flow identity test so the coordinate conversion can be verified before the expensive DINO run.
